# GraphRAG Evaluation Pipeline - ContextWeaver
## RAGAS + GraphRAG-Bench Metrics (arXiv 2506.05690)

Stack:
- Dataset: RAGAS testset ODS (Polish)
- Graph: Neo4j `neo4j://localhost:7474`
- LLM: `gpt-40-mini` (OpenAI key from env)
- Embeddings: `text-embedding-3-small
- QA service: `POST /qa?rag_mode={GRAPH|VANILLA|HYBRID}`

Metrics:
| Category | Metrics |
|-----------|---------|
| Graph Quality (§3.3 + Table 11) | Node/Edge Count, Avg Degree, Avg Clustering, Prop. Non-isolated, Component Size |
| Retrieval (§3.3) | Context Relevance Eq.(3), Evidence Recall Eq.(4) |
| Generation (App. D) | ROUGE-L, Answer Accuracy Eq.(5-6), Faithfulness Eq.(7), Evidence Coverage Eq.(8) |
| RAGAS | Faithfulness, ResponseRelevancy, ContextPrecision, ContextRecall, FactualCorrectness, SemanticSimilarity |


In [ ]:
# %pip install ragas networkx rouge-score sentence-transformers openai neo4j pandas odfpy matplotlib seaborn

In [ ]:
import os, re, ast, math, json, asyncio, warnings
from pathlib import Path
from typing import List, Dict, Optional

import httpx
import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from dotenv import load_dotenv
from rouge_score import rouge_scorer
from sentence_transformers import util
from neo4j import GraphDatabase

from ragas import evaluate, EvaluationDataset
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import (
    Faithfulness, ResponseRelevancy,
    LLMContextPrecisionWithReference, LLMContextRecall,
    FactualCorrectness, SemanticSimilarity, RougeScore,
)
from ragas.llms import llm_factory
from ragas.embeddings import LangchainEmbeddingsWrapper

warnings.filterwarnings("ignore")
load_dotenv()

## Configuration

In [ ]:
import tiktoken

TESTSET_PATH = Path("magisterka_eval/tags/testset_clean.csv")

NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]
NEO4J_DATABASE = "neo4j"

QA_BASE_URL = "http://localhost:8080"
QA_ENDPOINT = f"{QA_BASE_URL}/qa"

os.environ.get("OPENAI_API_KEY")

LLM_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"
MAX_EMB_TOKENS = 8000
_enc = tiktoken.encoding_for_model(EMBEDDING_MODEL)

## Loading models

In [ ]:
from openai import OpenAI
from ragas.embeddings import OpenAIEmbeddings
from langchain_openai import OpenAIEmbeddings as LCOpenAIEmbeddings

# LLM (RAGAS + LLM-based scoring)
evaluator_llm = llm_factory(LLM_MODEL, client=OpenAI())
print(f"LLM: {LLM_MODEL}")


lc_emb = OpenAIEmbeddings(model=EMBEDDING_MODEL, client=OpenAI())
evaluator_emb = LangchainEmbeddingsWrapper(LCOpenAIEmbeddings(model=EMBEDDING_MODEL))
print("Embeddings wrapper ready")

In [ ]:
def truncate_for_embedding(text: str, max_tokens: int = MAX_EMB_TOKENS) -> str:
    tokens = _enc.encode(text)
    if len(tokens) <= max_tokens:
        return text
    return _enc.decode(tokens[:max_tokens])

def embed_texts_safe(texts: List[str]) -> list:
    truncated = [truncate_for_embedding(t) for t in texts if t and t.strip()]
    if not truncated:
        return []
    return lc_emb.embed_texts(truncated)

## Loading the RAGAS dataset (ODS)

Columns from `TestsetGenerator`: `user_input`, `reference_contexts`, `reference`


In [ ]:
def parse_contexts(raw) -> List[str]:
    if isinstance(raw, list):
        return [str(x) for x in raw if x]
    if isinstance(raw, str):
        raw = raw.strip()

        try:
            parsed = json.loads(raw)
            if isinstance(parsed, list):
                return [str(x) for x in parsed if x]
        except Exception:
            pass

        try:
            parsed = ast.literal_eval(raw)
            if isinstance(parsed, list):
                return [str(x) for x in parsed if x]
        except Exception:
            pass

        return [raw] if raw else []
    return []


df_raw = pd.read_csv(TESTSET_PATH)
print(f"Loaded {len(df_raw)} rows | columns: {list(df_raw.columns)}")
df_raw.head(3)

In [ ]:
# Column normalization ────────────────────────────────────────────────────────
col_map = {}
for col in df_raw.columns:
    lc = col.lower().strip()
    if "user_input" in lc or lc == "question":
        col_map["user_input"] = col
    elif "reference_context" in lc:
        col_map["reference_contexts"] = col
    elif lc == "reference" or "ground_truth" in lc:
        col_map["reference"] = col

print("Column mapping:", col_map)
assert all(k in col_map for k in ["user_input", "reference_contexts", "reference"]), (
    f"Missing columns! Found: {list(df_raw.columns)}"
)

RAW_SAMPLES = []
for _, row in df_raw.iterrows():
    question = str(row[col_map["user_input"]]).strip()
    reference = str(row[col_map["reference"]]).strip()
    ref_contexts = parse_contexts(row[col_map["reference_contexts"]])
    if question and reference:
        RAW_SAMPLES.append({
            "question": question,
            "reference": reference,
            "reference_contexts": ref_contexts,
        })

print(f"Loaded {len(RAW_SAMPLES)} samples")
print(f"\nExample [0]:")
print(f"  question   : {RAW_SAMPLES[0]['question'][:80]}...")
print(f"  reference  : {RAW_SAMPLES[0]['reference'][:80]}...")
print(f"  contexts   : {len(RAW_SAMPLES[0]['reference_contexts'])} fragments")

## Calling the QA service

For each question we call the `POST /qa?rag_mode=X` endpoint in 3 modes:
`GRAPH_CHUNK`, `VANILLA`, `HYBRID`


In [ ]:
async def call_qa(question: str, rag_mode: str, timeout: float = 600.0) -> Dict:
    """Calls POST /qa?rag_mode=<mode> and returns a dict with the answer."""
    payload = {"question": question}
    async with httpx.AsyncClient(timeout=timeout) as client:
        resp = await client.post(
            QA_ENDPOINT,
            json=payload,
            params={"rag_mode": rag_mode},
        )
        resp.raise_for_status()
        data = resp.json()

    answer = data.get("answer")
    retrieved = data.get("source_context")
    return {"response": answer, "retrieved_contexts": retrieved, "raw": data}


async def run_all_modes(samples: List[Dict], modes=("vanilla", "hybrid", "graph")) -> Dict[str, List]:
    """Calls the service for each sample x each mode."""
    results = {mode: [] for mode in modes}
    total = len(samples) * len(modes)
    done = 0
    for s in samples:
        for mode in modes:
            try:
                out = await call_qa(s["question"], mode)
            except Exception as e:
                print(f"  [{mode}] error for '{s['question'][:40]}': {e}")
                out = {"response": "", "retrieved_contexts": [], "raw": {}}
            results[mode].append({
                **s,
                "response": out["response"],
                "retrieved_contexts": out["retrieved_contexts"],
                "rag_mode": mode,
            })
            done += 1
            if done % 5 == 0:
                print(f"  [{done}/{total}] ...")
    return results


print("Calling the QA service for all samples x 3 modes...")
qa_results = await run_all_modes(RAW_SAMPLES)
print(f"\nDone!")
for mode, rows in qa_results.items():
    non_empty = sum(1 for r in rows if r["response"])
    print(f"  {mode:8}: {non_empty}/{len(rows)} answers")

In [ ]:
for mode, rows in qa_results.items():
    print(f"\n=== {mode} ===")
    for r in rows:
        ctx = r.get("retrieved_contexts")
        print(f"  question : {r['question'][:50]}")
        print(f"  response : {repr(r['response'])}")
        print(f"  ctx type : {type(ctx)}, len: {len(ctx) if ctx else 'N/A'}")
        if ctx:
            print(f"  ctx   : {repr(ctx)[:80]}")

In [ ]:
from datetime import datetime
import csv


def save_qa_results_to_csv(qa_results: Dict[str, List], output_path: str):
    rows = []
    for mode, samples in qa_results.items():
        for r in samples:
            ctx = r.get("retrieved_contexts", "")
            if isinstance(ctx, list):
                ctx = "\n---\n".join(str(c) for c in ctx if c)

            rows.append({
                "rag_mode":           mode,
                "question":           r.get("question", ""),
                "response":           r.get("response", ""),
                "reference":          r.get("reference", ""),
                "retrieved_contexts": ctx,
                "reference_contexts": "\n---\n".join(r.get("reference_contexts", []) or []),
            })

    df = pd.DataFrame(rows)
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)

    df.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig",
        quoting=csv.QUOTE_ALL,
    )
    print(f"Saved {len(df)} rows → {output_path}")

timestamp = datetime.now().strftime("%Y%m%d_%H%M")
save_qa_results_to_csv(
    qa_results,
    output_path=f"eval/results/qa_results_{timestamp}.csv"
)

## Loading the graph from Neo4j

In [ ]:
def neo4j_to_networkx(uri: str, user: str, password: str, database: str) -> nx.DiGraph:
    """
    Fetches all nodes and relationships from Neo4j and builds an nx.DiGraph.
    Node/edge properties are kept as attributes.
    """
    G = nx.DiGraph()
    driver = GraphDatabase.driver(uri, auth=(user, password))
    try:
        with driver.session(database=database) as session:
            # Nodes
            node_result = session.run(
                "MATCH (n) RETURN id(n) AS nid, labels(n) AS labels, properties(n) AS props"
            )
            for record in node_result:
                nid = record["nid"]
                G.add_node(
                    nid,
                    labels=record["labels"],
                    **{k: v for k, v in record["props"].items()},
                )
            print(f"  Loaded {G.number_of_nodes()} nodes")

            # Edges
            edge_result = session.run(
                "MATCH (a)-[r]->(b) "
                "RETURN id(a) AS src, id(b) AS dst, type(r) AS rel_type, properties(r) AS props"
            )
            for record in edge_result:
                G.add_edge(
                    record["src"], record["dst"],
                    relation=record["rel_type"],
                    **{k: v for k, v in record["props"].items()},
                )
            print(f"  Loaded {G.number_of_edges()} edges")
    finally:
        driver.close()
    return G


print(f"Connecting to Neo4j ({NEO4J_URI})...")
G = neo4j_to_networkx(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD, NEO4J_DATABASE)
print(f"Graph loaded: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# Preview of nodes with the highest degree
top_nodes = sorted(dict(G.degree()).items(), key=lambda x: x[1], reverse=True)[:10]
print("Top-10 nodes (degree):")
for nid, deg in top_nodes:
    label = G.nodes[nid].get("id") or G.nodes[nid].get("name") or str(nid)
    print(f"  [{deg:3}]  {str(label)[:60]}")

## Graph Quality Metrics
*Section 3.3 + Appendix C Table 11 — arXiv 2506.05690*


In [ ]:
def compute_graph_quality(G: nx.DiGraph, corpus_tokens: Optional[int] = None) -> Dict:
    """
    Graph metrics from arXiv 2506.05690:
    - Eq. (1): Average Degree
    - Eq. (2): Average Clustering Coefficient
    - Table 11: Corpus stats
    """
    UG = G.to_undirected()
    N  = UG.number_of_nodes()
    E  = UG.number_of_edges()
    if N == 0:
        raise ValueError("Empty graph!")

    degrees = dict(UG.degree())

    # Eq.(1) Average Degree
    avg_degree = sum(degrees.values()) / N

    # Eq.(2) Average Clustering Coefficient
    avg_clust = nx.average_clustering(UG)

    non_iso = [v for v, d in degrees.items() if d > 0]
    prop_non_iso = len(non_iso) / N
    prop_deg = {k: sum(1 for d in degrees.values() if d > k) / N for k in [1, 2, 3]}

    components = list(nx.connected_components(UG))
    comp_sizes = [len(c) for c in components]
    geo_mean = math.exp(sum(math.log(s) for s in comp_sizes) / len(comp_sizes))

    lcc = max(nx.connected_components(UG), key=len)
    lcc_sub = UG.subgraph(lcc)
    try:
        diameter = nx.diameter(lcc_sub)
        avg_sp = nx.average_shortest_path_length(lcc_sub)
    except Exception:
        diameter = avg_sp = None

    return {
        "node_count": N,
        "edge_count": E,
        "average_degree": round(avg_degree, 4),
        "avg_clustering_coeff": round(avg_clust, 4),
        "prop_non_isolated": round(prop_non_iso, 4),
        "prop_degree_gt_1": round(prop_deg[1], 4),
        "prop_degree_gt_2":       round(prop_deg[2], 4),
        "prop_degree_gt_3": round(prop_deg[3], 4),
        "avg_component_size_geo": round(geo_mean, 4),
        "num_components": len(components),
        "lcc_size": len(lcc),
        "lcc_ratio": round(len(lcc)/N, 4),
        "graph_density": round(nx.density(UG), 6),
        "orphan_ratio": round(1 - prop_non_iso, 4),
        "max_degree": max(degrees.values()),
        "diameter_lcc": diameter,
        "avg_shortest_path_lcc": round(avg_sp, 4) if avg_sp else None,
        # Per 1k tok
        "avg_entities_per_1k": round(N / (corpus_tokens/1000), 4) if corpus_tokens else None,
        "avg_relations_per_1k": round(E / (corpus_tokens/1000), 4) if corpus_tokens else None,
    }


all_texts = [ctx for s in RAW_SAMPLES for ctx in s["reference_contexts"]]
corpus_tokens_est = int(sum(len(t.split()) * 1.3 for t in all_texts))

graph_metrics = compute_graph_quality(G, corpus_tokens=corpus_tokens_est)

print("=" * 58)
print("  GRAPH QUALITY METRICS  (arXiv 2506.05690)")
print("=" * 58)
for k, v in graph_metrics.items():
    bar = ""
    if isinstance(v, float) and 0 < v <= 1:
        bar = " " + "█" * int(v * 20)
    print(f"  {k:<35} {str(v):>10}{bar}")


## Retrieval Metrics (Paper Eq. 3 & 4)

Uses the embedding model.


In [ ]:
RETRIEVAL_THRESHOLD = 0.65   # threshold for Evidence Recall
RELEVANCE_THRESHOLD_HIGH = 0.72
RELEVANCE_THRESHOLD_MID = 0.45

def score_to_rating(score: float) -> float:
    if score >= RELEVANCE_THRESHOLD_HIGH: return 1.0
    elif score >= RELEVANCE_THRESHOLD_MID: return 0.5
    return 0.0

def compute_context_relevance(question: str, retrieved_contexts: List[str]) -> Dict:
    """Context Relevance — Eq.(3) from arXiv 2506.05690."""
    if isinstance(retrieved_contexts, str):
        retrieved_contexts = [retrieved_contexts] if retrieved_contexts.strip() else []
    if not retrieved_contexts:
        return {"context_relevance": 0.0, "r1": 0.0, "r2": 0.0}

    valid_contexts = [c for c in retrieved_contexts if c and c.strip()]
    if not valid_contexts:
        return {"context_relevance": 0.0, "r1": 0.0, "r2": 0.0}

    q_emb = embed_texts_safe([question])
    c_emb = embed_texts_safe(valid_contexts)
    scores = util.cos_sim(q_emb, c_emb)[0].tolist()
    top2 = sorted(scores, reverse=True)[:2]
    while len(top2) < 2: top2.append(0.0)
    r1, r2 = score_to_rating(top2[0]), score_to_rating(top2[1])
    return {
        "context_relevance": round((r1 + r2) / 2, 4),
        "r1": r1, "r2": r2,
        "top_scores": [round(s, 4) for s in top2],
    }

def compute_evidence_recall(reference_contexts: List[str], retrieved_contexts: List[str]) -> Dict:
    """Evidence Recall — Eq.(4) from arXiv 2506.05690."""
    if isinstance(retrieved_contexts, str):
        retrieved_contexts = [retrieved_contexts] if retrieved_contexts.strip() else []

    reference_contexts = [c for c in reference_contexts if c and c.strip()]
    retrieved_contexts = [c for c in retrieved_contexts if c and c.strip()]

    if not reference_contexts:
        return {"evidence_recall": 0.0, "matched": 0, "total": 0}
    if not retrieved_contexts:
        return {"evidence_recall": 0.0, "matched": 0, "total": len(reference_contexts)}
    ref_emb = embed_texts_safe(reference_contexts)
    ret_emb = embed_texts_safe(retrieved_contexts)
    sim = util.cos_sim(ref_emb, ret_emb)
    matched = sum(1 for i in range(len(reference_contexts))
                  if sim[i].max().item() >= RETRIEVAL_THRESHOLD)
    return {
        "evidence_recall": round(matched / len(reference_contexts), 4),
        "matched": matched,
        "total": len(reference_contexts),
    }

retrieval_dfs = {}
for mode, rows in qa_results.items():
    results = []
    for r in rows:
        cr = compute_context_relevance(r["question"], r["retrieved_contexts"])
        er = compute_evidence_recall(r["reference_contexts"], r["retrieved_contexts"])
        results.append({
            "question": r["question"][:55] + "...",
            "mode":     mode,
            **cr, **er,
        })
    retrieval_dfs[mode] = pd.DataFrame(results)

print("=" * 65)
print("  RETRIEVAL METRICS — Avg per mode")
print("=" * 65)
print(f"  {'Mode':<12} {'Context Relevance':>20} {'Evidence Recall':>18}")
print("  " + "-"*52)
for mode, df in retrieval_dfs.items():
    print(f"  {mode:<12} {df['context_relevance'].mean():>20.4f} {df['evidence_recall'].mean():>18.4f}")

## Generation Accuracy Metrics (App. D — Eq. 5–8)


In [ ]:
CLAIM_THRESHOLD   = 0.62
COVERAGE_THRESHOLD = 0.58

_rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)


def atomic_claims(text: str) -> List[str]:
    parts = re.split(r"[.;!?]", text)
    return [p.strip() for p in parts if len(p.strip()) > 8]


def rouge_l(pred: str, ref: str) -> float:
    return round(_rouge.score(ref, pred)["rougeL"].fmeasure, 4)


def semantic_similarity(response: str, reference: str) -> float:
    """SS = cos(f_i, c_j) — Eq.(6)."""
    embs = embed_texts_safe([response, reference])
    return round(util.cos_sim(embs[0], embs[1]).item(), 4)


def factual_correctness(response: str, reference: str) -> Dict:
    """FC = 2*TP/(TP+FP+FN) — Eq.(6)."""
    rc = atomic_claims(response)
    rf = atomic_claims(reference)
    if not rc or not rf:
        return {"factual_correctness": 0.0, "TP": 0, "FP": len(rc), "FN": len(rf)}
    re_embs = embed_texts_safe(rc)
    rf_embs = embed_texts_safe(rf)
    sim = util.cos_sim(re_embs, rf_embs)
    TP = sum(1 for i in range(len(rc)) if sim[i].max().item() >= CLAIM_THRESHOLD)
    FP = len(rc) - TP
    FN = sum(1 for j in range(len(rf)) if sim[:, j].max().item() < CLAIM_THRESHOLD)
    fc = 2*TP / (TP + FP + FN) if (TP + FP + FN) > 0 else 0.0
    return {"factual_correctness": round(fc, 4), "TP": TP, "FP": FP, "FN": FN}


def answer_accuracy(response: str, reference: str, alpha: float = 0.5) -> Dict:
    """AC = alpha*FC + (1-alpha)*SS — Eq.(5)."""
    fc = factual_correctness(response, reference)
    ss = semantic_similarity(response, reference)
    ac = alpha * fc["factual_correctness"] + (1 - alpha) * ss
    return {"answer_accuracy": round(ac, 4), "factual_correctness": fc["factual_correctness"],
            "semantic_similarity": ss}


def faithfulness_score(response: str, retrieved_contexts: List[str]) -> Dict:
    """FS = |{c∈A: S(c,C)}|/|A| — Eq.(7)."""
    if isinstance(retrieved_contexts, str):
        retrieved_contexts = [retrieved_contexts] if retrieved_contexts.strip() else []

    claims = atomic_claims(response)
    if not claims or not retrieved_contexts:
        return {"faithfulness_paper": 0.0, "supported": 0, "total_claims": len(claims)}
    ctx_emb = embed_texts_safe([" ".join(retrieved_contexts)])
    cl_embs = embed_texts_safe(claims)
    sims = util.cos_sim(cl_embs, ctx_emb)[:, 0].tolist()
    supported = sum(1 for s in sims if s >= CLAIM_THRESHOLD)
    return {"faithfulness_paper": round(supported/len(claims), 4),
            "supported": supported, "total_claims": len(claims)}


def evidence_coverage(response: str, reference_contexts: List[str]) -> Dict:
    """Cov = |{e∈E: M(e,G)}|/|E| — Eq.(8)."""
    if not reference_contexts:
        return {"evidence_coverage": 0.0}
    resp_emb = embed_texts_safe([response])
    ref_embs = embed_texts_safe(reference_contexts)
    sims = util.cos_sim(ref_embs, resp_emb)[:, 0].tolist()
    covered = sum(1 for s in sims if s >= COVERAGE_THRESHOLD)
    return {"evidence_coverage": round(covered/len(reference_contexts), 4),
            "covered": covered, "total": len(reference_contexts)}


# For each mode
generation_dfs = {}
for mode, rows in qa_results.items():
    gen_rows = []
    for r in rows:
        if not r["response"]:
            gen_rows.append({"question": r["question"][:55]+"...", "mode": mode,
                             "rouge_l": 0, "answer_accuracy": 0, "factual_correctness": 0,
                             "semantic_similarity": 0, "faithfulness_paper": 0, "evidence_coverage": 0})
            continue
        row = {"question": r["question"][:55]+"...", "mode": mode}
        row["rouge_l"] = rouge_l(r["response"], r["reference"])
        row.update(answer_accuracy(r["response"], r["reference"]))
        row.update(faithfulness_score(r["response"], r["retrieved_contexts"]))
        row.update(evidence_coverage(r["response"], r["reference_contexts"]))
        gen_rows.append(row)
    generation_dfs[mode] = pd.DataFrame(gen_rows)

# Summary table
summary_gen = pd.DataFrame([
    {
        "mode": mode,
        "rouge_l":            df["rouge_l"].mean(),
        "answer_accuracy":    df["answer_accuracy"].mean(),
        "factual_correctness":df["factual_correctness"].mean(),
        "semantic_similarity":df["semantic_similarity"].mean(),
        "faithfulness_paper": df["faithfulness_paper"].mean(),
        "evidence_coverage":  df["evidence_coverage"].mean(),
    }
    for mode, df in generation_dfs.items()
]).round(4)

print("=" * 85)
print("GENERATION METRICS - Avg per mode")
print("=" * 85)
print(summary_gen.to_string(index=False))

## RAGAS Metrics (per mode)

In [ ]:
def to_ctx_list(ctx) -> List[str]:
    if isinstance(ctx, list):
        return ctx if ctx else ["brak kontekstu"]
    if isinstance(ctx, str) and ctx.strip():
        return [ctx]
    return ["brak kontekstu"]

ragas_metric_list = [
    Faithfulness(llm=evaluator_llm),
    ResponseRelevancy(llm=evaluator_llm, embeddings=evaluator_emb),
    LLMContextPrecisionWithReference(llm=evaluator_llm),
    LLMContextRecall(llm=evaluator_llm),
    FactualCorrectness(llm=evaluator_llm),
    SemanticSimilarity(embeddings=evaluator_emb),
    RougeScore(),
]

ragas_dfs = {}
for mode, rows in qa_results.items():
    print(f"RAGAS [{mode}]...")
    samples_eval = [
        SingleTurnSample(
            user_input=r["question"],
            response=r["response"] or "brak odpowiedzi",
            reference=r["reference"],
            retrieved_contexts=to_ctx_list(r["retrieved_contexts"]),
            reference_contexts=to_ctx_list(r["reference_contexts"]),
        )
        for r in rows
    ]
    ds = EvaluationDataset(samples=samples_eval)
    result = evaluate(dataset=ds, metrics=ragas_metric_list)
    df_r = result.to_pandas()
    df_r["mode"] = mode
    ragas_dfs[mode] = df_r
    print(f"  {mode} OK")


first_df = next(iter(ragas_dfs.values()))
ragas_num_cols = [c for c in first_df.columns
                  if c not in ["user_input","response","retrieved_contexts",
                               "reference","reference_contexts","mode"]
                  and pd.api.types.is_numeric_dtype(first_df[c])]

summary_ragas = pd.DataFrame([
    {"mode": mode, **{col: df[col].mean() for col in ragas_num_cols}}
    for mode, df in ragas_dfs.items()
]).round(4)

print("\n" + "=" * 75)
print("RAGAS METRICS - Avg per mode")
print("=" * 75)
print(summary_ragas.to_string(index=False))

In [ ]:
import csv

all_ragas = pd.concat(ragas_dfs.values(), ignore_index=True)
all_ragas.to_csv(
    "Graph RAG Dataset/testset/ragas_detailed.csv",
    index=False,
    encoding="utf-8-sig",
    quoting=csv.QUOTE_ALL,
)

summary_ragas.to_csv(
    "Graph RAG Dataset/testset/ragas_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

print(f"Saved {len(all_ragas)} detailed rows + summary")

## Full Comparison Report (GRAPH vs VANILLA vs HYBRID)

In [ ]:
# All metrics ──────────────────────────────────────────────────
all_results = []
for mode in qa_results.keys():
    df_r = retrieval_dfs[mode].copy()
    df_g = generation_dfs[mode].copy()
    df_ra = ragas_dfs[mode].copy()
    base = df_r[["question","mode","context_relevance","evidence_recall"]].reset_index(drop=True)
    gen = df_g[["rouge_l","answer_accuracy","factual_correctness",
                   "semantic_similarity","faithfulness_paper","evidence_coverage"]].reset_index(drop=True)
    rag = df_ra[ragas_num_cols].add_prefix("ragas_").reset_index(drop=True)
    combined = pd.concat([base, gen, rag], axis=1)
    all_results.append(combined)

df_all = pd.concat(all_results, ignore_index=True)

# Pivot table: metric x mode ──────────────────────────────────────────────
metric_cols = [c for c in df_all.columns if c not in ["question", "mode"]]
pivot = df_all.groupby("mode")[metric_cols].mean().round(4).T
pivot.index.name = "Metric"

def source_of(m):
    if m in ["context_relevance","evidence_recall"]:
        return "Paper Retrieval"
    if m in ["rouge_l","answer_accuracy","factual_correctness",
              "semantic_similarity","faithfulness_paper","evidence_coverage"]:
        return "Paper Generation"
    return "RAGAS"

pivot.insert(0, "Source", [source_of(m) for m in pivot.index])
pivot = pivot.sort_values("Source")

print("=" * 90)
print("FULL COMPARISON: Paper + RAGAS  |  GRAPH_CHUNK vs VANILLA vs HYBRID")
print("=" * 90)
print(pivot.to_string())
df_all

## Visualizations

In [ ]:
MODES = list(qa_results.keys())
COLORS = {"graph_chunk": "#4C72B0", "vanilla": "#DD8452", "hybrid": "#55A868"}
fig = plt.figure(figsize=(22, 20))
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.48, wspace=0.38)

# Knowledge Graph (top-150 nodes) ──────────────────────────────────────
ax0 = fig.add_subplot(gs[0, :2])
G_vis = G.to_undirected()
# Top-N nodes for visualization
N_VIS = min(120, G_vis.number_of_nodes())
top_vis = sorted(dict(G_vis.degree()).items(), key=lambda x: x[1], reverse=True)[:N_VIS]
top_ids = [n for n, _ in top_vis]
subG = G_vis.subgraph(top_ids)
pos = nx.spring_layout(subG, seed=42, k=1.5)
degs = dict(subG.degree())
nx.draw_networkx_nodes(subG, pos, node_size=[100+80*degs[n] for n in subG.nodes()],
                        node_color=[degs[n] for n in subG.nodes()],
                        cmap=plt.cm.YlOrRd, alpha=0.85, ax=ax0)
nx.draw_networkx_edges(subG, pos, alpha=0.25, ax=ax0, arrows=False)
if N_VIS <= 40:
    labels_map = {n: (G.nodes[n].get("id") or G.nodes[n].get("name") or str(n))[:20]
                  for n in subG.nodes()}
    nx.draw_networkx_labels(subG, pos, labels=labels_map, font_size=6, ax=ax0)
ax0.set_title(
    f"Knowledge Graph (top-{N_VIS} nodes)  |  "
    f"N={graph_metrics['node_count']}  E={graph_metrics['edge_count']}  "
    f"avg_deg={graph_metrics['average_degree']}  "
    f"avg_clust={graph_metrics['avg_clustering_coeff']}",
    fontsize=9, fontweight="bold")
ax0.axis("off")

# Degree distribution ────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 2])
all_degs = sorted([d for _, d in G.to_undirected().degree()], reverse=True)
ax1.bar(range(min(60, len(all_degs))), all_degs[:60], color="#4C72B0", edgecolor="white", width=0.8)
ax1.axhline(graph_metrics["average_degree"], color="red", linestyle="--",
            label=f"avg={graph_metrics['average_degree']:.2f}")
ax1.set_title("Degree Distribution (top-60)", fontweight="bold", fontsize=9)
ax1.set_xlabel("Node rank"); ax1.set_ylabel("Degree"); ax1.legend(fontsize=8)

# Retrieval: Context Relevance per mode ──────────────────────────────────
ax2 = fig.add_subplot(gs[1, 0])
for mode in MODES:
    vals = retrieval_dfs[mode]["context_relevance"].values
    ax2.plot(range(len(vals)), vals, "o-", label=mode, color=COLORS[mode], markersize=4, alpha=0.8)
ax2.axhline(0.5, color="gray", linestyle="--", linewidth=0.8)
ax2.set_title("Context Relevance per sample", fontweight="bold", fontsize=9)
ax2.set_xlabel("Sample idx"); ax2.set_ylabel("Score"); ax2.legend(fontsize=8); ax2.set_ylim(-0.05, 1.15)

# Retrieval: Evidence Recall per mode ────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 1])
for mode in MODES:
    vals = retrieval_dfs[mode]["evidence_recall"].values
    ax3.plot(range(len(vals)), vals, "s-", label=mode, color=COLORS[mode], markersize=4, alpha=0.8)
ax3.axhline(0.5, color="gray", linestyle="--", linewidth=0.8)
ax3.set_title("Evidence Recall per sample", fontweight="bold", fontsize=9)
ax3.set_xlabel("Sample idx"); ax3.set_ylabel("Score"); ax3.legend(fontsize=8); ax3.set_ylim(-0.05, 1.15)

# Generation: radar/bar comparison ──────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 2])
gen_m = ["rouge_l","answer_accuracy","faithfulness_paper","evidence_coverage"]
x     = np.arange(len(gen_m))
w     = 0.25
for i, mode in enumerate(MODES):
    vals = [generation_dfs[mode][m].mean() for m in gen_m]
    ax4.bar(x + (i-1)*w, vals, w, label=mode, color=COLORS[mode], alpha=0.85)
ax4.set_xticks(x)
ax4.set_xticklabels(["ROUGE-L","Ans.Acc","Faithful","EvCov"], fontsize=8)
ax4.set_title("Generation Metrics (avg)", fontweight="bold", fontsize=9)
ax4.legend(fontsize=8); ax4.set_ylim(0, 1.2); ax4.set_ylabel("Score")

# RAGAS heatmap ──────────────────────────────────────────────────────────
ax5 = fig.add_subplot(gs[2, :2])
hm_data = summary_ragas.set_index("mode")[ragas_num_cols]
sns.heatmap(hm_data.T, annot=True, fmt=".3f", cmap="RdYlGn",
            vmin=0, vmax=1, ax=ax5, linewidths=0.5,
            annot_kws={"size": 9})
ax5.set_title("RAGAS Metrics Heatmap (GRAPH_CHUNK vs VANILLA vs HYBRID)", fontweight="bold", fontsize=10)
ax5.set_xlabel("RAG Mode"); ax5.set_ylabel("")
plt.setp(ax5.get_xticklabels(), rotation=0)

# Paper vs RAGAS equivalents ────────────────────────────────────────────
ax6 = fig.add_subplot(gs[2, 2])
paper_ragas = {
    "faithfulness (paper)":  ("faithfulness_paper",  "ragas_faithfulness"),
    "ev. recall (paper)":    ("evidence_recall",      "ragas_context_recall"),
    "rouge-l":                ("rouge_l",              "ragas_rouge_score"),
    "sem. sim.":              ("semantic_similarity",   "ragas_semantic_similarity"),
}
for ax_mode, (fld1, fld2) in paper_ragas.items():
    pass  # chart only

labels_, paper_vals, ragas_vals = [], [], []
for label, (pk, rk) in paper_ragas.items():
    col_paper = pk if pk in df_all.columns else None
    col_ragas = rk if rk in df_all.columns else None
    if col_paper and col_ragas:
        labels_.append(label)
        paper_vals.append(df_all[col_paper].mean())
        ragas_vals.append(df_all[col_ragas].mean())

x_ = np.arange(len(labels_))
ax6.bar(x_-0.2, paper_vals, 0.36, label="Paper (Stella)", color="#4C72B0", alpha=0.85)
ax6.bar(x_+0.2, ragas_vals, 0.36, label="RAGAS (LLM)",   color="#DD8452", alpha=0.85)
for i, (pv, rv) in enumerate(zip(paper_vals, ragas_vals)):
    ax6.text(i-0.2, pv+0.02, f"{pv:.2f}", ha="center", fontsize=8, color="#4C72B0", fontweight="bold")
    ax6.text(i+0.2, rv+0.02, f"{rv:.2f}", ha="center", fontsize=8, color="#DD8452", fontweight="bold")
ax6.set_xticks(x_); ax6.set_xticklabels(labels_, fontsize=8)
ax6.set_ylim(0, 1.2); ax6.set_title("Paper vs RAGAS (overall avg)", fontweight="bold", fontsize=9)
ax6.legend(fontsize=8)

fig.suptitle(
    "ContextWeaver - GraphRAG Evaluation Dashboard\n"
    f"Dataset: {len(RAW_SAMPLES)} Q&A  |  Graph: {graph_metrics['node_count']} nodes / {graph_metrics['edge_count']} edges  |  LLM: {LLM_MODEL}",
    fontsize=12, fontweight="bold", y=1.01,
)
plt.savefig("contextweaver_eval_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()
print("Dashboard: contextweaver_eval_dashboard.png")


## Graph vs Benchmarks from Table 11 (arXiv 2506.05690)

In [ ]:
TABLE_11 = pd.DataFrame({
    "Benchmark":          ["UltraDomain","MultiHop-RAG","HotpotQA",
                            "GraphRAG-Bench (novel)","GraphRAG-Bench (medical)",
                            f"ContextWeaver [{NEO4J_DATABASE}]"],
    "Avg Entities/1k":    [170.6, 10.1, 39.3, 19.6, 11.8, graph_metrics["avg_entities_per_1k"]],
    "Avg Relations/1k":   [73.2,  3.82, 12.7, 20.9,  6.2, graph_metrics["avg_relations_per_1k"]],
    "Prop Non-isolated":  [0.40,  0.41, 0.41, 0.66, 0.48, graph_metrics["prop_non_isolated"]],
    "Avg Degree":         [0.86,  0.76, 0.65, 2.27, 1.05, graph_metrics["average_degree"]],
    "Prop Degree>1":      [0.27,  0.26, 0.23, 0.47, 0.36, graph_metrics["prop_degree_gt_1"]],
    "Prop Degree>2":      [0.15,  0.14, 0.12, 0.25, 0.20, graph_metrics["prop_degree_gt_2"]],
    "Prop Degree>3":      [0.09,  0.09, 0.06, 0.17, 0.12, graph_metrics["prop_degree_gt_3"]],
    "Avg Comp Size":      [2.71,  2.70, 2.11, 3.99, 3.15, graph_metrics["avg_component_size_geo"]],
    "Avg Clust Coeff":    [None, None, None, None, None,   graph_metrics["avg_clustering_coeff"]],
}).set_index("Benchmark")

print("=" * 100)
print("GRAPH STRUCTURE vs TABLE 11 BENCHMARKS  (arXiv 2506.05690, Appendix C)")
print("=" * 100)

styled = TABLE_11.style.highlight_max(axis=0, props="font-weight:bold;color:darkgreen")                       .format("{:.3f}", na_rep="—")
display(styled)
TABLE_11

## Exporting results

In [ ]:
output_dir = TESTSET_PATH.parent
xlsx_path  = output_dir / "evaluation_results.xlsx"

with pd.ExcelWriter(xlsx_path) as writer:
    # Per-sample results for each mode
    for mode in MODES:
        df_r  = retrieval_dfs[mode]
        df_g  = generation_dfs[mode]
        df_ra = ragas_dfs[mode][["user_input"] + ragas_num_cols].rename(
            columns={c: "ragas_"+c for c in ragas_num_cols})
        full = pd.concat([
            df_r.reset_index(drop=True),
            df_g.drop(columns=["question","mode"], errors="ignore").reset_index(drop=True),
            df_ra.reset_index(drop=True),
        ], axis=1)
        full.to_excel(writer, sheet_name=f"mode_{mode}", index=False)

    pivot.to_excel(writer, sheet_name="Summary_pivot")
    summary_ragas.to_excel(writer, sheet_name="RAGAS_summary", index=False)
    summary_gen.to_excel(writer, sheet_name="Generation_summary", index=False)
    TABLE_11.to_excel(writer, sheet_name="Graph_vs_Table11")
    pd.DataFrame([graph_metrics]).T.rename(columns={0:"value"}).to_excel(
        writer, sheet_name="Graph_quality")

print(f"Results saved: {xlsx_path}")

# Final summary ──────────────────────────────────────────────────────────────
print()
print("=" * 65)
print("FINAL SUMMARY - ContextWeaver GraphRAG Evaluation")
print("=" * 65)
print(f"  Samples:       {len(RAW_SAMPLES)}")
print(f"  Modes:         {', '.join(MODES)}")
print(f"  Graph:         {graph_metrics['node_count']} nodes | {graph_metrics['edge_count']} edges")
print(f"  Avg Degree:    {graph_metrics['average_degree']}")
print(f"  Avg Clust:     {graph_metrics['avg_clustering_coeff']}")
print()
print(f"  {'Metric':<28} " + "  ".join(f"{m:>10}" for m in MODES))
print("  " + "-"*(28 + 14*len(MODES)))
all_metrics = (
    [("context_relevance", retrieval_dfs), ("evidence_recall", retrieval_dfs)]
    + [(m, generation_dfs) for m in ["rouge_l","answer_accuracy","faithfulness_paper","evidence_coverage"]]
)
for metric, dfs_dict in all_metrics:
    row = f"  {metric:<28} "
    for mode in MODES:
        row += f"  {dfs_dict[mode][metric].mean():>10.4f}"
    print(row)
print()
for col in ragas_num_cols:
    row = f"  ragas_{col:<22} "
    for mode in MODES:
        row += f"  {ragas_dfs[mode][col].mean():>10.4f}"
    print(row)